<a href="https://colab.research.google.com/github/DrDavidL/learning-dhds/blob/main/DHDS_Session1_Colab_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HS-DHDS Session 1: Introduction to AI Tools & Data Exploration

**Northwestern University Feinberg School of Medicine**  
**Digital Health and Data Science**  
**January 30, 2026**

---

## Welcome to Google Colab!

This notebook will guide you through:
1. Understanding how Colab works
2. Loading and exploring a synthetic pulmonary dataset
3. Creating visualizations (box plots, scatter plots, correlations)
4. Using the Gemini AI assistant for code help

**Important Reminder:** This is a cloud-based tool. Use only with synthetic or public data. Never upload PHI!

## Part 1: Getting Oriented with Colab

### Key Concepts
- **Cells**: Colab notebooks are made of cells. This is a *text cell* (Markdown).
- **Code cells**: Run Python code by clicking the play button or pressing `Shift+Enter`
- **Gemini**: Look for the Gemini icon in the right sidebar. You can ask it to help write or explain code!

### Try it: Run the cell below

In [ ]:
# This is a code cell. Click the play button or press Shift+Enter to run it.
print("Hello! You are ready for DHDS Session 1!")
print("Today's date:", "January 30, 2026")

## Part 2: Loading Our Synthetic Pulmonary Dataset

We will work with a **synthetic** dataset that mimics real pulmonary patient data. This allows us to learn data analysis techniques safely.

### What is Tidy Data?
- Each **variable** forms a column
- Each **observation** forms a row
- Each **observational unit** forms a table

Let's start by importing our libraries and creating the dataset.

In [ ]:
# Import libraries we will use
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set a nice style for our plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]

print("Libraries imported successfully!")

In [ ]:
# Generate synthetic pulmonary patient data
# This mimics real clinical data without any actual patient information

np.random.seed(42)  # For reproducibility
n_patients = 200

# Create synthetic data
data = {
    'patient_id': [f'PT{str(i).zfill(4)}' for i in range(1, n_patients + 1)],
    'age': np.random.normal(62, 12, n_patients).clip(30, 90).astype(int),
    'sex': np.random.choice(['Male', 'Female'], n_patients, p=[0.55, 0.45]),
    'smoking_status': np.random.choice(['Never', 'Former', 'Current'], n_patients, p=[0.25, 0.45, 0.30]),
    'pack_years': np.zeros(n_patients),
}

# Assign pack_years based on smoking status
for i in range(n_patients):
    if data['smoking_status'][i] == 'Never':
        data['pack_years'][i] = 0
    elif data['smoking_status'][i] == 'Former':
        data['pack_years'][i] = np.random.normal(25, 15, 1)[0].clip(5, 60)
    else:  # Current
        data['pack_years'][i] = np.random.normal(35, 15, 1)[0].clip(10, 80)

# Create DataFrame
df = pd.DataFrame(data)

# Generate FEV1 and FVC based on age, sex, and smoking
# These formulas are simplified approximations for educational purposes
base_fev1 = 4.0 - (df['age'] - 30) * 0.03
base_fvc = 5.0 - (df['age'] - 30) * 0.025

# Adjust for sex
df['FEV1'] = np.where(df['sex'] == 'Female', base_fev1 * 0.78, base_fev1)
df['FVC'] = np.where(df['sex'] == 'Female', base_fvc * 0.80, base_fvc)

# Reduce lung function based on pack-years
df['FEV1'] = df['FEV1'] - (df['pack_years'] * 0.015)
df['FVC'] = df['FVC'] - (df['pack_years'] * 0.008)

# Add some random variation
df['FEV1'] = df['FEV1'] + np.random.normal(0, 0.3, n_patients)
df['FVC'] = df['FVC'] + np.random.normal(0, 0.35, n_patients)

# Ensure reasonable values
df['FEV1'] = df['FEV1'].clip(0.8, 5.0).round(2)
df['FVC'] = df['FVC'].clip(1.0, 6.0).round(2)

# Calculate FEV1/FVC ratio
df['FEV1_FVC_ratio'] = (df['FEV1'] / df['FVC']).round(3)

# Determine COPD diagnosis (simplified: FEV1/FVC < 0.70)
df['COPD_diagnosis'] = (df['FEV1_FVC_ratio'] < 0.70).astype(int)

# Round pack_years
df['pack_years'] = df['pack_years'].round(1)

print(f"Created synthetic dataset with {len(df)} patients")
print(f"\nThis is TIDY DATA: each row is one patient, each column is one variable.")

## Part 3: Exploring the Data

Before doing any analysis, we should understand our data. This is called **Exploratory Data Analysis (EDA)**.

In [ ]:
# View the first few rows
print("First 10 rows of our dataset:")
df.head(10)

In [ ]:
# Check the structure of our data
print("Dataset Info:")
print(f"Number of patients: {len(df)}")
print(f"Number of variables: {len(df.columns)}")
print(f"\nColumn names and types:")
print(df.dtypes)

In [ ]:
# Summary statistics for numeric variables
print("Summary Statistics:")
df.describe().round(2)

In [ ]:
# Check the distribution of categorical variables
print("Smoking Status Distribution:")
print(df['smoking_status'].value_counts())
print(f"\nSex Distribution:")
print(df['sex'].value_counts())
print(f"\nCOPD Diagnosis:")
print(df['COPD_diagnosis'].value_counts())
print(f"\n{df['COPD_diagnosis'].mean()*100:.1f}% of patients have COPD diagnosis")

## Part 4: Data Visualization

Visualizations help us see patterns that numbers alone might miss. Let's create:
1. **Histograms** - see distributions
2. **Box plots** - compare groups
3. **Scatter plots** - see relationships
4. **Correlation matrix** - see all relationships at once

### 4.1 Histograms: Distribution of FEV1

In [ ]:
# Histogram of FEV1 values
plt.figure(figsize=(10, 5))
plt.hist(df['FEV1'], bins=25, edgecolor='black', color='steelblue', alpha=0.7)
plt.xlabel('FEV1 (Liters)', fontsize=12)
plt.ylabel('Number of Patients', fontsize=12)
plt.title('Distribution of FEV1 Values in Our Pulmonary Dataset', fontsize=14)
plt.axvline(df['FEV1'].mean(), color='red', linestyle='--', label=f'Mean: {df["FEV1"].mean():.2f}L')
plt.legend()
plt.tight_layout()
plt.show()

### 4.2 Box Plots: Comparing Groups

Box plots show:
- The **median** (middle line)
- The **interquartile range** (box = 25th to 75th percentile)
- **Outliers** (points beyond the whiskers)

In [ ]:
# Box plot: FEV1 by Smoking Status
plt.figure(figsize=(10, 6))
order = ['Never', 'Former', 'Current']
sns.boxplot(x='smoking_status', y='FEV1', data=df, order=order, palette='Set2')
plt.xlabel('Smoking Status', fontsize=12)
plt.ylabel('FEV1 (Liters)', fontsize=12)
plt.title('FEV1 by Smoking Status', fontsize=14)
plt.tight_layout()
plt.show()

# Print the means for each group
print("Mean FEV1 by Smoking Status:")
print(df.groupby('smoking_status')['FEV1'].mean().round(2))

In [ ]:
# Box plot: FEV1/FVC ratio by COPD diagnosis
plt.figure(figsize=(8, 6))
sns.boxplot(x='COPD_diagnosis', y='FEV1_FVC_ratio', data=df, palette=['lightgreen', 'salmon'])
plt.xlabel('COPD Diagnosis (0 = No, 1 = Yes)', fontsize=12)
plt.ylabel('FEV1/FVC Ratio', fontsize=12)
plt.title('FEV1/FVC Ratio by COPD Diagnosis', fontsize=14)
plt.axhline(y=0.70, color='red', linestyle='--', label='Diagnostic Threshold (0.70)')
plt.legend()
plt.tight_layout()
plt.show()

### 4.3 Scatter Plots: Relationships Between Variables

Scatter plots show how two continuous variables relate to each other.

In [ ]:
# Scatter plot: Age vs FEV1
plt.figure(figsize=(10, 6))
colors = {'Male': 'steelblue', 'Female': 'coral'}
for sex in ['Male', 'Female']:
    subset = df[df['sex'] == sex]
    plt.scatter(subset['age'], subset['FEV1'], alpha=0.6, label=sex, c=colors[sex])

plt.xlabel('Age (years)', fontsize=12)
plt.ylabel('FEV1 (Liters)', fontsize=12)
plt.title('FEV1 Decline with Age', fontsize=14)
plt.legend()

# Add trend line
z = np.polyfit(df['age'], df['FEV1'], 1)
p = np.poly1d(z)
plt.plot(df['age'].sort_values(), p(df['age'].sort_values()), "r--", alpha=0.8, label='Trend')

plt.tight_layout()
plt.show()

print(f"For every year of age, FEV1 decreases by approximately {-z[0]:.3f} liters")

In [ ]:
# Scatter plot: Pack-years vs FEV1
plt.figure(figsize=(10, 6))
colors_copd = {0: 'green', 1: 'red'}
labels_copd = {0: 'No COPD', 1: 'COPD'}

for copd in [0, 1]:
    subset = df[df['COPD_diagnosis'] == copd]
    plt.scatter(subset['pack_years'], subset['FEV1'], alpha=0.6,
                label=labels_copd[copd], c=colors_copd[copd])

plt.xlabel('Pack-Years', fontsize=12)
plt.ylabel('FEV1 (Liters)', fontsize=12)
plt.title('Relationship Between Smoking History and FEV1', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

### 4.4 Correlation Matrix

A correlation matrix shows the correlation coefficient between all pairs of numeric variables.

**Remember:** Correlation ranges from -1 to +1
- **+1**: Perfect positive relationship
- **0**: No linear relationship
- **-1**: Perfect negative relationship

**Important:** Correlation does NOT imply causation!

In [ ]:
# Select numeric columns for correlation
numeric_cols = ['age', 'pack_years', 'FEV1', 'FVC', 'FEV1_FVC_ratio', 'COPD_diagnosis']
correlation_matrix = df[numeric_cols].corr().round(2)

# Create heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, fmt='.2f',
            annot_kws={'size': 11})
plt.title('Correlation Matrix: Pulmonary Dataset Variables', fontsize=14)
plt.tight_layout()
plt.show()

print("\nKey Observations:")
print(f"- FEV1 and FVC are highly correlated: r = {correlation_matrix.loc['FEV1', 'FVC']:.2f}")
print(f"- Pack-years negatively correlates with FEV1: r = {correlation_matrix.loc['pack_years', 'FEV1']:.2f}")
print(f"- Age negatively correlates with FEV1: r = {correlation_matrix.loc['age', 'FEV1']:.2f}")

## Part 5: Using Gemini to Help with Code

Look for the **Gemini icon** in the right sidebar of Colab. You can ask it questions like:

- "How do I create a bar chart showing the count of patients by smoking status?"
- "Explain what this code does"
- "How do I filter the dataframe to only show patients with COPD?"

### Try it yourself!

In the cell below, try writing code with Gemini's help. Some ideas:
1. Create a bar chart of smoking status counts
2. Filter to show only current smokers
3. Calculate the mean FEV1 for patients over 65

In [ ]:
# YOUR TURN: Try writing code here with Gemini's help!
# Click the Gemini icon in the right sidebar and ask for help.

# Example prompt to try: "Create a bar chart showing the number of patients by smoking status"



## Part 6: Key Takeaways

### What We Learned Today

1. **Tidy Data**: One row per observation, one column per variable
2. **Exploratory Data Analysis**: Always explore before modeling
3. **Visualizations**:
   - Histograms show distributions
   - Box plots compare groups
   - Scatter plots show relationships
   - Correlation matrices show all relationships
4. **Correlation is not causation**: A relationship does not prove cause and effect
5. **AI Assistance**: Gemini can help write and explain code, but you must critically evaluate outputs

### Core Principle

**If you cannot verify it independently, do not use it clinically.**

### Next Session Preview

In Session 2, we will:
- Generate synthetic data for specific research questions
- Build predictive models (e.g., COPD prediction)
- Learn about model evaluation (AUROC, calibration)
- Discuss algorithmic bias in healthcare AI

## Resources

- **Session Materials**: github.com/DrDavidL/learning-dhds
- **AutoAnalyzer (NU NetID)**: autoanalyze.azurewebsites.net
- **NM Secure Chat (PHI-approved)**: chat.nm.org
- **Questions**: DavidL@northwestern.edu